![](https://i.ibb.co/7t6wKN7/000.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Hello friends 🙃</b>
    <br>There is nothing to learn in this notebook, but here you can find a class that converts the dataset from the organizers into COCO format with just one call. This way you can just copy the class and use it without going into details. I hope this will be helpful<br>
</div>

![](https://thumbs.gfycat.com/JampackedBothJumpingbean-size_restricted.gif)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Structure of the COCO Dataset</b>
    <br>In the image below you can see the structure of the dataset that you will get after using the class, as well as a small legend for the schema 👇<br>
</div>

![](https://i.ibb.co/jzPRck5/Screenshot-from-2023-06-25-06-02-15.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Libraries ⚡️</b>
</div>

In [1]:
import itertools
from tqdm import tqdm
import yaml

import numpy as np
from sklearn.model_selection import train_test_split

import json
from pathlib import Path
import os
import shutil

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Class for easy converting to COCO format 🤌🏻</b>
</div>

In [2]:
class ConvertToCOCO:
    def __init__(self,
                 dataset_dirpath: str,
                 new_dataset_dirpath: str,
                 annotation_path: str,
                 train_size: float = 0.80,
                 random_seed: int = 42
                 ):
        assert train_size >= 0 or train_size <= 1
        self.train_size = train_size
        self.random_seed = random_seed
        self.id_dict = {
            "blood_vessel": 0,
            "glomerulus": 1,
            "unsure": 2
        }

        self.dirpath = Path(dataset_dirpath)
        self.__samples = self.__parse_jsonl(annotation_path)
        self.new_dataset_dirpath = Path(os.path.join(new_dataset_dirpath, "dataset"))
        self.dataset_config_path = Path(os.path.join(self.new_dataset_dirpath, "coco.yaml"))
        self.train_dirpath =  Path(os.path.join(self.new_dataset_dirpath, "train"))
        self.val_dirpath = Path(os.path.join(self.new_dataset_dirpath, "val"))

    @staticmethod
    def __parse_jsonl(path: str) -> list[dict, ...]:
        with open(path, 'r') as json_file:
            jsonl_labels = [
                json.loads(line)
                for line in tqdm(json_file)
            ]
        return jsonl_labels

    def split_dataset(self) -> tuple:
        train_samples, val_samples = train_test_split(
            self.__samples,
            train_size=self.train_size,
            random_state=self.random_seed
        )
        return train_samples, val_samples

    def mkdirs(self) -> None:
        os.mkdir(self.new_dataset_dirpath)
        os.mkdir(self.train_dirpath)
        os.mkdir(self.val_dirpath)

    def to_coco(self, tile: list[dict, ...], output_folder: Path) -> None:
        tile_id = tile["id"]
        shutil.copyfile(self.dirpath / f"train/{tile_id}.tif", output_folder / f"{tile_id}.tif")
        with open(output_folder / f"{tile_id}.txt", "w") as text_file:
            for annotation in tile["annotations"]:
                class_id = self.id_dict[annotation["type"]]
                flat_mask_polygon = list(itertools.chain(*annotation["coordinates"][0]))
                array = np.array(flat_mask_polygon) / 512.
                text_file.write(f'{class_id} {" ".join(map(str, array))}\n')

    def convert(self, samples: list, path: Path) -> None:
        for data in tqdm(samples):
            self.to_coco(data, path)

    def get_config(self):
        return {
            "train": str(self.train_dirpath),
            "val": str(self.val_dirpath),
            "names": [
                {0: "blood_vessel"},
                {1: "glomerulus"},
                {2: "unsure"}
            ]
        }

    def write_config(self, config: dict) -> None:
        with open(self.dataset_config_path, mode="w") as f:
            yaml.safe_dump(stream=f, data=config)

    def __call__(self, make_config: bool = True) -> None:
        self.mkdirs()
        train_samples, val_samples = self.split_dataset()
        self.convert(train_samples, self.train_dirpath)
        self.convert(val_samples, self.val_dirpath)
        if make_config:
            config = self.get_config()
            self.write_config(config)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>And this is what the data configuration file looks like. It is very important if you are going to use a framework like ultralytics 👇</b>
</div>

![](https://i.ibb.co/wLrD9Dt/Screenshot-from-2023-06-25-07-35-26.png)

<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>Let's check🏻🚦</b>
</div>

In [3]:
coco = ConvertToCOCO(
    dataset_dirpath="/kaggle/input/hubmap-hacking-the-human-vasculature",
    new_dataset_dirpath="/kaggle/working",
    annotation_path="/kaggle/input/hubmap-hacking-the-human-vasculature/polygons.jsonl",
)

1633it [00:03, 516.28it/s]


In [4]:
coco()

100%|██████████| 327/327 [00:04<00:00, 69.29it/s]


<div class="alert alert-block alert-info" style="font-size:20px; background-color: #f4ffab; font-family:verdana; color: #8a0f0f; border: 3px #cbd600 solid">
    <b>I hope this was helpful 😀 Should I make a notepad using this dataset in practice? Good luck!</b>
</div>